In [ ]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import torch
import torchvision.transforms as transforms

model_window = None
saved_mask_image = None
loaded_image_path = None

device = "cuda" if torch.cuda.is_available() else "cpu"

def preprocess_image(img_path, target_size=(256, 256)):
    img = Image.open(img_path).convert("RGB")
    transform_infer = transforms.Compose([
        transforms.Resize(target_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    return transform_infer(img).unsqueeze(0).to(device), img

def tensor_to_pil(tensor):
    tensor = tensor.squeeze(0).cpu()
    tensor = (tensor * 0.5 + 0.5).clamp(0, 1)
    return transforms.ToPILImage()(tensor)

def set_background(window, bg_image_path):
    try:
        bg_image = Image.open(bg_image_path)
        bg_image = bg_image.resize((window.winfo_screenwidth(), window.winfo_screenheight()), Image.LANCZOS)
        bg_photo = ImageTk.PhotoImage(bg_image)
        window.bg_image = bg_photo
        bg_label = tk.Label(window, image=bg_photo)
        bg_label.image = bg_photo
        bg_label.place(x=0, y=0, relwidth=1, relheight=1)
    except Exception as e:
        print(f"Error loading background image: {e}")

def create_model_window(model_path, bg_image_path, title):
    global model_window, saved_mask_image, loaded_image_path
    root.withdraw()

    gen_AB = torch.jit.load(model_path, map_location=device)
    gen_AB.eval()

    def show_main_window():
        global model_window
        if model_window:
            model_window.destroy()
        root.deiconify()

    def load_image():
        file_path = filedialog.askopenfilename()
        if file_path:
            global loaded_image_path
            loaded_image_path = file_path
            img = Image.open(file_path).convert("RGB").resize((256, 256))
            photo = ImageTk.PhotoImage(img)
            label_image.config(image=photo)
            label_image.image = photo
            button_save.config(state=tk.DISABLED)

    def apply_model():
        global saved_mask_image
        if loaded_image_path is not None:
            try:
                input_tensor, orig_img = preprocess_image(loaded_image_path)
                with torch.no_grad():
                    output_tensor = gen_AB(input_tensor)
                output_img = tensor_to_pil(output_tensor)
                saved_mask_image = output_img

                photo_mask = ImageTk.PhotoImage(output_img)
                label_mask.config(image=photo_mask)
                label_mask.image = photo_mask
                button_save.config(state=tk.NORMAL)
            except Exception as e:
                print(f"Error applying model: {e}")

    def save_image():
        if saved_mask_image:
            file_path = filedialog.asksaveasfilename(
                defaultextension=".png",
                filetypes=[("PNG files", "*.png"), ("All files", "*.*")]
            )
            if file_path:
                saved_mask_image.save(file_path)
                print(f"Image saved at {file_path}")

    model_window = tk.Toplevel()
    model_window.title(title)
    set_background(model_window, bg_image_path)

    label_title = tk.Label(model_window, text=title, font=('Arial', 18, 'bold'), bg='#ffffff', fg='#000000')
    label_title.pack(pady=15)

    button_back = tk.Button(
        model_window,
        text="Back to Control Panel",
        command=show_main_window,
        bg='#007BFF',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_back.pack(pady=15)

    frame_buttons = tk.Frame(model_window, bg='')
    frame_buttons.pack(pady=0)

    button_load = tk.Button(
        frame_buttons,
        text="Load Image",
        command=load_image,
        bg='#28A745',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_load.pack(side='left', padx=0, pady=0)

    button_apply = tk.Button(
        frame_buttons,
        text="Apply Model",
        command=apply_model,
        bg='#DC3545',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_apply.pack(side='left', padx=0, pady=0)

    button_save = tk.Button(
        frame_buttons,
        text="Save Image",
        command=save_image,
        bg='#17A2B8',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8,
        state=tk.DISABLED
    )
    button_save.pack(side='left', padx=0, pady=0)

    frame_image = tk.Frame(model_window, bg='')
    frame_image.pack(pady=15)

    label_image = tk.Label(frame_image)
    label_image.pack()

    frame_mask = tk.Frame(model_window, bg='')
    frame_mask.pack(pady=15)

    label_mask = tk.Label(frame_mask)
    label_mask.pack()

    screen_width = model_window.winfo_screenwidth()
    screen_height = model_window.winfo_screenheight()
    model_window.geometry(f"{screen_width}x{screen_height}")

root = tk.Tk()
root.title("Control Panel")
set_background(root, 'background.jpg')

label_main_title = tk.Label(
    root,
    text="Main Control Interface",
    font=('Arial', 24, 'bold'),
    bg='#ffffff',
    fg='#000000'
)
label_main_title.pack(pady=20)

frame_buttons = tk.Frame(root, bg='', padx=20, pady=20)
frame_buttons.place(relx=0.5, rely=0.5, anchor="center")

button_model1 = tk.Button(
    frame_buttons,
    text="Model 1 - Convert to Nature Images",
    command=lambda: create_model_window('gen_AB2_torchscript.pt', 'background.jpg', 'Model 1 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model1.pack(pady=10)

button_model2 = tk.Button(
    frame_buttons,
    text="Model 2 - Convert to Monet Style",
    command=lambda: create_model_window('gen_BA2_torchscript.pt', 'background.jpg', 'Model 2 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model2.pack(pady=10)

button_model3 = tk.Button(
    frame_buttons,
    text="Model 3 - Convert to Colored Images",
    command=lambda: create_model_window('gen_AB3_torchscript.pt', 'background.jpg', 'Model 3 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model3.pack(pady=10)

button_model4 = tk.Button(
    frame_buttons,
    text="Model 4 - Convert to Grayscale",
    command=lambda: create_model_window('gen_BA3_torchscript.pt', 'background.jpg', 'Model 4 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model4.pack(pady=10)

button_exit = tk.Button(
    frame_buttons,
    text="Exit",
    command=root.quit,
    bg='#DC3545',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_exit.pack(pady=10)

screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
root.geometry(f"{screen_width}x{screen_height}")

root.mainloop()

In [ ]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import torch
import torchvision.transforms as transforms

model_window = None
saved_mask_image = None
loaded_image_path = None

device = "cuda" if torch.cuda.is_available() else "cpu"

def preprocess_image(img_path, target_size=(256, 256)):
    img = Image.open(img_path).convert("RGB")
    transform_infer = transforms.Compose([
        transforms.Resize(target_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    return transform_infer(img).unsqueeze(0).to(device), img

def tensor_to_pil(tensor):
    tensor = tensor.squeeze(0).cpu()
    tensor = (tensor * 0.5 + 0.5).clamp(0, 1)
    return transforms.ToPILImage()(tensor)

def set_background(window, bg_image_path):
    try:
        bg_image = Image.open(bg_image_path)
        bg_image = bg_image.resize((window.winfo_screenwidth(), window.winfo_screenheight()), Image.LANCZOS)
        bg_photo = ImageTk.PhotoImage(bg_image)
        window.bg_image = bg_photo
        bg_label = tk.Label(window, image=bg_photo)
        bg_label.image = bg_photo
        bg_label.place(x=0, y=0, relwidth=1, relheight=1)
    except Exception as e:
        print(f"Error loading background image: {e}")

def create_model_window(model_path, bg_image_path, title):
    global model_window, saved_mask_image, loaded_image_path
    root.withdraw()

    gen_AB = torch.jit.load(model_path, map_location=device)
    gen_AB.eval()

    def show_main_window():
        global model_window
        if model_window:
            model_window.destroy()
        root.deiconify()

    def load_image():
        file_path = filedialog.askopenfilename()
        if file_path:
            global loaded_image_path
            loaded_image_path = file_path
            img = Image.open(file_path).convert("RGB").resize((256, 256))
            photo = ImageTk.PhotoImage(img)
            label_image.config(image=photo)
            label_image.image = photo
            button_save.config(state=tk.DISABLED)

    def apply_model():
        global saved_mask_image
        if loaded_image_path is not None:
            try:
                input_tensor, orig_img = preprocess_image(loaded_image_path)
                with torch.no_grad():
                    output_tensor = gen_AB(input_tensor)
                output_img = tensor_to_pil(output_tensor)
                saved_mask_image = output_img

                photo_mask = ImageTk.PhotoImage(output_img)
                label_mask.config(image=photo_mask)
                label_mask.image = photo_mask
                button_save.config(state=tk.NORMAL)
            except Exception as e:
                print(f"Error applying model: {e}")

    def save_image():
        if saved_mask_image:
            file_path = filedialog.asksaveasfilename(
                defaultextension=".png",
                filetypes=[("PNG files", "*.png"), ("All files", "*.*")]
            )
            if file_path:
                saved_mask_image.save(file_path)
                print(f"Image saved at {file_path}")

    model_window = tk.Toplevel()
    model_window.title(title)
    set_background(model_window, bg_image_path)

    label_title = tk.Label(model_window, text=title, font=('Arial', 18, 'bold'), bg='#ffffff', fg='#000000')
    label_title.pack(pady=15)

    button_back = tk.Button(
        model_window,
        text="Back to Control Panel",
        command=show_main_window,
        bg='#007BFF',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_back.pack(pady=15)

    frame_buttons = tk.Frame(model_window, bg='')
    frame_buttons.pack(pady=0)

    button_load = tk.Button(
        frame_buttons,
        text="Load Image",
        command=load_image,
        bg='#28A745',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_load.pack(side='left', padx=0, pady=0)

    button_apply = tk.Button(
        frame_buttons,
        text="Apply Model",
        command=apply_model,
        bg='#DC3545',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8
    )
    button_apply.pack(side='left', padx=0, pady=0)

    button_save = tk.Button(
        frame_buttons,
        text="Save Image",
        command=save_image,
        bg='#17A2B8',
        fg='white',
        font=('Arial', 12, 'bold'),
        relief='flat',
        padx=15,
        pady=8,
        state=tk.DISABLED
    )
    button_save.pack(side='left', padx=0, pady=0)

    frame_image = tk.Frame(model_window, bg='')
    frame_image.pack(pady=15)

    label_image = tk.Label(frame_image)
    label_image.pack()

    frame_mask = tk.Frame(model_window, bg='')
    frame_mask.pack(pady=15)

    label_mask = tk.Label(frame_mask)
    label_mask.pack()

    screen_width = model_window.winfo_screenwidth()
    screen_height = model_window.winfo_screenheight()
    model_window.geometry(f"{screen_width}x{screen_height}")

root = tk.Tk()
root.title("Control Panel")
set_background(root, 'background.jpg')

label_main_title = tk.Label(
    root,
    text="Main Control Interface",
    font=('Arial', 24, 'bold'),
    bg='#ffffff',
    fg='#000000'
)
label_main_title.pack(pady=20)

frame_buttons = tk.Frame(root, bg='', padx=20, pady=20)
frame_buttons.place(relx=0.5, rely=0.5, anchor="center")

button_model1 = tk.Button(
    frame_buttons,
    text="Model 1 - Convert to Nature Images",
    command=lambda: create_model_window('gen_AB2_torchscript.pt', 'background.jpg', 'Model 1 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model1.pack(pady=10)

button_model2 = tk.Button(
    frame_buttons,
    text="Model 2 - Convert to Monet Style",
    command=lambda: create_model_window('gen_BA2_torchscript.pt', 'background.jpg', 'Model 2 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model2.pack(pady=10)

button_model3 = tk.Button(
    frame_buttons,
    text="Model 3 - Convert to Colored Images",
    command=lambda: create_model_window('gen_AB3_torchscript.pt', 'background.jpg', 'Model 3 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model3.pack(pady=10)

button_model4 = tk.Button(
    frame_buttons,
    text="Model 4 - Convert to Grayscale",
    command=lambda: create_model_window('gen_BA3_torchscript.pt', 'background.jpg', 'Model 4 Interface'),
    bg='#007BFF',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_model4.pack(pady=10)

button_exit = tk.Button(
    frame_buttons,
    text="Exit",
    command=root.quit,
    bg='#DC3545',
    fg='white',
    font=('Arial', 12, 'bold'),
    relief='flat',
    padx=15,
    pady=8
)
button_exit.pack(pady=10)

screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
root.geometry(f"{screen_width}x{screen_height}")

root.mainloop()